In [1]:
!pip install pandas requests tqdm


[notice] A new release of pip is available: 23.2.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import requests
from tqdm import tqdm
import os

In [3]:
PROMPT_TEMPLATE_1 = """
Instruction: Generate ONLY a JSON output following this schema:
```json
{{ 
  "review": "<review>" 
}}
```

Input:
{content}

Expected output (only JSON, no extra text):
"""

In [4]:
PROMPT_TEMPLATE_2 = """
Instruction: Analyze the following review text and provide the outputs formatted in JSON:

1. **Sentiment Classification:** Indicate whether the sentiment of the review is "positive" or "negative".

Generate ONLY a JSON output following this schema:
```json
{{
  "review": "<original_review>",
  "sentiment": "<sentiment>"
}}
```

Input:
{content}

Expected output (only JSON, no extra text):
"""

In [5]:
PROMPT_TEMPLATE_3 = """
Instruction: Analyze the following review text and provide the outputs formatted in JSON:

1. **Sentiment Classification:** Indicate whether the sentiment of the review is "positive" or "negative".
2. **Emotion Classification:** Indicate whether the emotion categories of the review is  "joy", "anger", "sadness", "disgust", "neutral".

Generate ONLY a JSON output following this schema:
```json
{{
  "review": "<original_review>",
  "sentiment": "<sentiment>",
  "emotion": "<emotion_categories>"
}}
```

Input:
{content}

Expected output (only JSON, no extra text):
"""

In [6]:
PROMPT_TEMPLATE_4 = """
Instruction: Analyze the following review text and provide the outputs formatted in JSON:

1. **Sentiment Classification:** Indicate whether the sentiment of the review is "positive" or "negative".
2. **Emotion Classification:** Indicate whether the emotion categories of the review is  "joy", "anger", "sadness", "disgust", "neutral".
3. **Topic Classification:** Indicate whether the topic categories of the review is  "action", "comedy", "drama", "horror", "thriller", "romance", "sci-fi", "fantasy", "documentary", "other".

Generate ONLY a JSON output following this schema:
```json
{{
  "review": "<original_review>",
  "sentiment": "<sentiment>",
  "emotion": "<emotion_categories>",
  "topic": "<topic_categories>"
}}
```

Input:
{content}

Expected output (only JSON, no extra text):
"""

In [7]:
PROMPT_TEMPLATE_5 = """
Instruction: Analyze the following review text and provide the outputs formatted in JSON:

1. **Sentiment Classification:** Indicate whether the sentiment of the review is "positive" or "negative".
2. **Emotion Classification:** Indicate whether the emotion categories of the review is  "joy", "anger", "sadness", "disgust", "neutral".
3. **Topic Classification:** Indicate whether the topic categories of the review is  "action", "comedy", "drama", "horror", "thriller", "romance", "sci-fi", "fantasy", "documentary", "other".
4. **Named Entity Extraction:** List all named entities present in the text, categorizing them by label (PERSON, ORG, LOC).

Generate ONLY a JSON output following this schema:
```json
{{
  "review": "<original_review>",
  "sentiment": "<sentiment>",
  "emotion": "<emotion_categories>",
  "topic": "<topic_categories>",
  "entities": [
    {{
      "label": "<label>",
      "value": "<value>"
    }}
  ]
}}
```

Input:
{content}

Expected output (only JSON, no extra text):
"""

In [8]:
PROMPT_TEMPLATE_6 = """
Instruction: Analyze the following review text and provide the outputs formatted in JSON:

1. **Sentiment Classification:** Indicate whether the sentiment of the review is "positive" or "negative".
2. **Emotion Classification:** Indicate whether the emotion categories of the review is  "joy", "anger", "sadness", "disgust", "neutral".
3. **Topic Classification:** Indicate whether the topic categories of the review is  "action", "comedy", "drama", "horror", "thriller", "romance", "sci-fi", "fantasy", "documentary", "other".
4. **Named Entity Extraction:** List all named entities present in the text, categorizing them by label (PERSON, ORG, LOC).
5. **Translation:** Translate the review from English to Italian.

Generate ONLY a JSON output following this schema:
```json
{{
  "review": "<original_review>",
  "sentiment": "<sentiment>",
  "emotion": "<emotion_categories>",
  "topic": "<topic_categories>",
  "entities": [
    {{
      "label": "<label>",
      "value": "<value>"
    }}
  ],
  "translate": "<italian_translation>"
}}
```

Input:
{content}

Expected output (only JSON, no extra text):
"""

In [9]:
PROMPT_TEMPLATES = {
    1: PROMPT_TEMPLATE_1,
    2: PROMPT_TEMPLATE_2,
    3: PROMPT_TEMPLATE_3,
    4: PROMPT_TEMPLATE_4,
    5: PROMPT_TEMPLATE_5,
    6: PROMPT_TEMPLATE_6,
}

TASK_NAMES = {
    1: "json_only",
    2: "json_sentiment", 
    3: "json_sentiment_emotion",
    4: "json_sentiment_emotion_topic",
    5: "json_sentiment_emotion_topic_ner",
    6: "json_sentiment_emotion_topic_ner_translation"
}

# Define the model name to use
MODEL_NAME = "qwen3:4b-instruct"

# Define the input file path
INPUT_FILE_PATH = "../resources/IMDB Dataset 500 Sampled With Translate Emotion Topic Updated.csv"

print(f"✓ Configurazione completata:")
print(f"  - Modello: {MODEL_NAME}")
print(f"  - File input: {INPUT_FILE_PATH}")
print(f"  - Task configurati: {len(TASK_NAMES)}")

✓ Configurazione completata:
  - Modello: qwen3:4b-instruct
  - File input: ../resources/IMDB Dataset 500 Sampled With Translate Emotion Topic Updated.csv
  - Task configurati: 6


In [10]:
def process_review(review: str, model_name: str, prompt_template: str) -> str:
    """
    It processes the review text and returns the LLM response as string.
    
    Arguments:
        review (str): The review text.
        model_name (str): The model name for Ollama.
        prompt_template (str): The prompt template to use.
        
    Return:
        The LLM response as string.
    """
    try:
        return requests.post(
            url="http://localhost:11434/api/generate",
            json={
                "model": model_name,
                "prompt": prompt_template.format(content=review),
                "stream": False
            }
        ).json()["response"]
    except Exception as e:
        print(f"Error invoking the chain: {str(e)}")
        return None

In [11]:
def call_model_llm(model_name: str, task_number: int, input_file_path: str) -> None:
    """
    It calls the LLM model using Ollama for a specific task. 
    
    Arguments:
        model_name: The name of the model to invoke via Ollama.
        task_number: Which task to run (1-6).
        input_file_path: Path to the input CSV file.
    
    Return:
        None (saves results to file).
    """
    # Get the appropriate prompt template
    prompt_template = PROMPT_TEMPLATES[task_number]
    task_name = TASK_NAMES[task_number]
    
    # Define output file path (replace : with _ for Windows compatibility)
    model_name_clean = model_name.replace(':', '_')
    output_file_path = f"../Output Performance Degradation Analysis/{model_name_clean}_six_task_invert/task_{task_number}_{task_name}_{model_name_clean}.csv"

    # Create output directory if it doesn't exist
    output_dir = os.path.dirname(output_file_path)
    os.makedirs(output_dir, exist_ok=True)
    
    # Check if output file exists, if not create it
    if not os.path.exists(output_file_path):
        sampled = pd.read_csv(input_file_path)
        sampled["output"] = sampled.apply(lambda row: "$$$", axis=1)
        sampled.to_csv(output_file_path, index=False)
        print(f"✓ Created new output file: {output_file_path}")
    else:
        print(f"✓ Using existing output file: {output_file_path}")
    
    # Load the dataframe
    dataframe = pd.read_csv(output_file_path)
    already_done_part = dataframe[~(dataframe.output == "$$$")].copy()
    slice_to_work_on = dataframe[dataframe.output == "$$$"].copy()
    slice_to_work_on.reset_index(inplace=True, drop=True)
    total_rows = len(slice_to_work_on)
    
    print(f"Processing Task {task_number} ({task_name}) with {total_rows} reviews...")
    
    for i in tqdm(range(total_rows), total=total_rows, desc=f"Task {task_number}"):
        row = slice_to_work_on.iloc[i]
        result = process_review(row["review"], model_name, prompt_template)
        slice_to_work_on.loc[i, "output"] = result
        updated_df = pd.concat([already_done_part, slice_to_work_on])
        updated_df.to_csv(output_file_path, index=False)
    
    print(f"✓ Task {task_number} ({task_name}) completato!")

print("✓ Funzione call_model_llm definita")

✓ Funzione call_model_llm definita


In [12]:
print("="*60)
print("TASK 1: JSON formatting")
print("="*60)

call_model_llm(model_name=MODEL_NAME, task_number=1, input_file_path=INPUT_FILE_PATH)

TASK 1: JSON formatting
✓ Using existing output file: ../Output Performance Degradation Analysis/qwen3_4b-instruct_six_task_invert/task_1_json_only_qwen3_4b-instruct.csv
Processing Task 1 (json_only) with 223 reviews...


Task 1: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 223/223 [1:31:08<00:00, 24.52s/it]

✓ Task 1 (json_only) completato!


In [13]:
print("="*60)
print("TASK 2: JSON + Sentiment")
print("="*60)

call_model_llm(model_name=MODEL_NAME, task_number=2, input_file_path=INPUT_FILE_PATH)

TASK 2: JSON + Sentiment
✓ Created new output file: ../Output Performance Degradation Analysis/qwen3_4b-instruct_six_task_invert/task_2_json_sentiment_qwen3_4b-instruct.csv
Processing Task 2 (json_sentiment) with 500 reviews...


Task 2: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 500/500 [3:29:46<00:00, 25.17s/it]

✓ Task 2 (json_sentiment) completato!


In [14]:
print("="*60)
print("TASK 3: JSON + Sentiment + Emotion")
print("="*60)

call_model_llm(model_name=MODEL_NAME, task_number=3, input_file_path=INPUT_FILE_PATH)

TASK 3: JSON + Sentiment + Emotion
✓ Created new output file: ../Output Performance Degradation Analysis/qwen3_4b-instruct_six_task_invert/task_3_json_sentiment_emotion_qwen3_4b-instruct.csv
Processing Task 3 (json_sentiment_emotion) with 500 reviews...


Task 3: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 500/500 [3:30:28<00:00, 25.26s/it]

✓ Task 3 (json_sentiment_emotion) completato!


In [12]:
print("="*60)
print("TASK 4: JSON + Sentiment + Emotion + Topic")
print("="*60)
call_model_llm(model_name=MODEL_NAME, task_number=4, input_file_path=INPUT_FILE_PATH)

TASK 4: JSON + Sentiment + Emotion + Topic
✓ Using existing output file: ../Output Performance Degradation Analysis/qwen3_4b-instruct_six_task_invert/task_4_json_sentiment_emotion_topic_qwen3_4b-instruct.csv
Processing Task 4 (json_sentiment_emotion_topic) with 41 reviews...


Task 4: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 41/41 [20:23<00:00, 29.85s/it]

✓ Task 4 (json_sentiment_emotion_topic) completato!


In [12]:
print("="*60)
print("TASK 5: JSON + Sentiment + Emotion + Topic + NER")
print("="*60)
call_model_llm(model_name=MODEL_NAME, task_number=5, input_file_path=INPUT_FILE_PATH)

TASK 5: JSON + Sentiment + Emotion + Topic + NER
✓ Using existing output file: ../Output Performance Degradation Analysis/qwen3_4b-instruct_six_task_invert/task_5_json_sentiment_emotion_topic_ner_qwen3_4b-instruct.csv
Processing Task 5 (json_sentiment_emotion_topic_ner) with 414 reviews...


Task 5: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 414/414 [1:55:17<00:00, 16.71s/it]

✓ Task 5 (json_sentiment_emotion_topic_ner) completato!


In [13]:
print("="*60)
print("TASK 6: JSON + Sentiment + Emotion + Topic + NER + Translation")
print("="*60)
call_model_llm(model_name=MODEL_NAME, task_number=6, input_file_path=INPUT_FILE_PATH)

TASK 6: JSON + Sentiment + Emotion + Topic + NER + Translation
✓ Created new output file: ../Output Performance Degradation Analysis/qwen3_4b-instruct_six_task_invert/task_6_json_sentiment_emotion_topic_ner_translation_qwen3_4b-instruct.csv
Processing Task 6 (json_sentiment_emotion_topic_ner_translation) with 500 reviews...


Task 6: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 500/500 [6:05:22<00:00, 43.84s/it]

✓ Task 6 (json_sentiment_emotion_topic_ner_translation) completato!
